In [2]:
# ============================================================
# Cell 1 — Load a combined sample across all traffic types
# ============================================================
import pandas as pd
import numpy as np

base = "/Users/abhulimen/Downloads/Group stuff fro capstone project/datasets/packetFeatures_datasets/"

files = {
    "benign": base + "BenignTraffic.csv",
    "benign_1": base + "BenignTraffic1.csv",
    "benign_2": base + "BenignTraffic2.csv",
    "benign_3": base + "BenignTraffic3.csv",
    "ddos": base + "DDoS-HTTP_Flood-.csv",
    "dos": base + "DoS-HTTP_Flood.csv",
    "dos_1": base + "DoS-HTTP_Flood1.csv",
    "dns_spoofing": base + "DNS_Spoofing.csv",
    "brute_force": base + "DictionaryBruteForce.csv",
    "xss": base + "XSS.csv",
}

SAMPLE_ROWS_PER_FILE = 10000

id_cols = [
    "stream", "src_mac", "dst_mac", "src_ip", "dst_ip",
    "src_port", "dst_port", "device_mac", "eth_src_oui", "eth_dst_oui",
]

dfs = []
for label, path in files.items():
    df = pd.read_csv(path, nrows=SAMPLE_ROWS_PER_FILE)
    df["source_label"] = label
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(f"Combined sample shape: {data.shape}")

numeric_df = data.select_dtypes(include=[np.number]).drop(
    columns=[c for c in id_cols if c in data.columns], errors="ignore"
)
print(f"Numeric feature columns: {numeric_df.shape[1]}")




Combined sample shape: (100000, 136)
Numeric feature columns: 116


In [3]:
# ============================================================
# Cell 2 — Correlation matrix + list of high-correlation pairs
# ============================================================
corr_matrix = numeric_df.corr()

THRESHOLD = 0.9

pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_matrix.iloc[i, j]
        if pd.notna(r) and abs(r) >= THRESHOLD:
            pairs.append((cols[i], cols[j], round(r, 4)))

pairs_df = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "correlation"])
pairs_df = pairs_df.sort_values("correlation", key=abs, ascending=False)

print(f"Found {len(pairs_df)} pairs with |correlation| >= {THRESHOLD}")
pairs_df.head(20)




Found 176 pairs with |correlation| >= 0.9


,feature_1,feature_2,correlation
102,src_ip_10_var,src_ip_mac_10_var,1.0000
48,src_ip_5_count,src_ip_mac_5_count,1.0000
163,src_ip_60_mean,src_ip_mac_60_mean,1.0000
16,src_ip_1_var,src_ip_mac_1_var,1.0000
164,src_ip_60_var,src_ip_mac_60_var,1.0000
143,src_ip_30_var,src_ip_mac_30_var,1.0000
166,min_et,min_e,1.0000
162,src_ip_60_count,src_ip_mac_60_count,1.0000
168,q1,q1_e,1.0000
97,src_ip_10_mean,src_ip_mac_10_mean,1.0000


In [4]:
# ============================================================
# Cell 3 — Rule-based redundant-column drop
#
# Rule: for features that come in multiple time-window variants
# (_1, _5, _10, _30, _60 seconds), prefer keeping the SMALLER window
# when two windowed features are highly correlated. Smaller windows
# react faster to sudden traffic bursts (e.g. DDoS/DoS floods), which
# matters more for an IDS than a smoothed-out longer window.
#
# For features with no window suffix (e.g. ttl, payload_entropy),
# there's no window preference — they're just processed in whatever
# order they appear, since there's no principled reason to prefer one
# over another without more domain knowledge.
#
# Mechanism: sort columns so smaller-window features are checked
# FIRST. In the greedy pass, whichever column is checked first
# becomes the one that's kept when a high-correlation pair is found.
# ============================================================
import re

def extract_window(col: str):
    """Return the time-window size (seconds) embedded in a column
    name, or None if the column has no window suffix."""
    match = re.search(r"_(1|5|10|30|60)_", col)
    return int(match.group(1)) if match else None


def window_priority(col: str):
    """Sort key: no-window columns first (priority -1, order doesn't
    matter for them), then windowed columns smallest-window-first."""
    window = extract_window(col)
    return -1 if window is None else window


def get_redundant_columns(corr_matrix: pd.DataFrame, threshold: float = 0.9) -> set:
    # Process smaller windows before larger ones, so smaller windows
    # are the ones kept when a correlated pair is found.
    cols = sorted(corr_matrix.columns, key=window_priority)
    to_drop = set()

    for i in range(len(cols)):
        col_i = cols[i]
        if col_i in to_drop:
            continue
        for j in range(i + 1, len(cols)):
            col_j = cols[j]
            if col_j in to_drop:
                continue
            r = corr_matrix.loc[col_i, col_j]
            if pd.notna(r) and abs(r) >= threshold:
                to_drop.add(col_j)

    return to_drop


redundant_cols = get_redundant_columns(corr_matrix, threshold=THRESHOLD)
print(f"Columns marked as redundant ({len(redundant_cols)}):")
print(sorted(redundant_cols))

reduced_numeric_df = numeric_df.drop(columns=redundant_cols)
print(f"\nNumeric features before: {numeric_df.shape[1]}")
print(f"Numeric features after:  {reduced_numeric_df.shape[1]}")
print(f"\nRemaining features ({reduced_numeric_df.shape[1]}):")
print(sorted(reduced_numeric_df.columns))




Columns marked as redundant (59):
['average_p', 'channel_10_count', 'channel_10_mean', 'channel_1_count', 'channel_1_var', 'channel_30_mean', 'channel_30_var', 'channel_5_mean', 'channel_5_var', 'channel_60_count', 'channel_60_mean', 'channel_60_var', 'icmp_checksum_status', 'min_e', 'min_p', 'payload_length', 'q1_e', 'q1_p', 'q3_p', 'src_ip_10_count', 'src_ip_10_mean', 'src_ip_10_var', 'src_ip_30_mean', 'src_ip_5_mean', 'src_ip_60_count', 'src_ip_60_mean', 'src_ip_60_var', 'src_ip_mac_10_count', 'src_ip_mac_10_mean', 'src_ip_mac_10_var', 'src_ip_mac_1_count', 'src_ip_mac_1_mean', 'src_ip_mac_1_var', 'src_ip_mac_30_count', 'src_ip_mac_30_mean', 'src_ip_mac_30_var', 'src_ip_mac_5_count', 'src_ip_mac_5_mean', 'src_ip_mac_5_var', 'src_ip_mac_60_count', 'src_ip_mac_60_mean', 'src_ip_mac_60_var', 'stream_10_count', 'stream_10_mean', 'stream_10_var', 'stream_30_var', 'stream_5_mean', 'stream_60_count', 'stream_60_mean', 'stream_60_var', 'stream_jitter_10_mean', 'stream_jitter_10_sum', 'strea

In [5]:
# ============================================================
# Cell 4 — Sanity check: re-run correlation on the reduced set
# to confirm nothing above threshold remains
# ============================================================
check_corr = reduced_numeric_df.corr()
remaining = []
cols = check_corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = check_corr.iloc[i, j]
        if pd.notna(r) and abs(r) >= THRESHOLD:
            remaining.append((cols[i], cols[j], round(r, 4)))

print(f"Remaining high-correlation pairs after drop: {len(remaining)}")
if remaining:
    print(remaining)




Remaining high-correlation pairs after drop: 0


In [6]:
# ============================================================
# Cell 5 — Low-variance / mostly-empty feature check
#
# Goal: find features that carry almost no information because
# they're nearly constant (e.g. a flag that's 0 for 99.9% of rows)
# or mostly missing/placeholder values (e.g. -1, "none", NaN).
# These don't need labels to evaluate, so they're safe to check
# even before deciding on the unsupervised model itself.
#
# Two checks:
#   1. Variance relative to the feature's own scale (a raw variance
#      number isn't comparable across features with very different
#      units/ranges, so we normalize it).
#   2. "Dominant value" ratio: what fraction of rows share the single
#      most common value. A feature that's the same value 99%+ of
#      the time isn't going to help separate benign vs. attack traffic.
# ============================================================
VARIANCE_THRESHOLD = 0.01     # normalized variance below this = flagged
DOMINANT_VALUE_THRESHOLD = 0.99  # one value makes up >=99% of rows = flagged

variance_report = []
for col in reduced_numeric_df.columns:
    series = reduced_numeric_df[col].dropna()
    if series.empty:
        variance_report.append((col, np.nan, 1.0, "all missing"))
        continue

    # normalized variance = std / mean magnitude, guards against
    # comparing raw variances across very different feature scales
    mean_abs = series.abs().mean()
    std = series.std()
    normalized_variance = std / mean_abs if mean_abs > 0 else 0.0

    dominant_ratio = series.value_counts(normalize=True).iloc[0]

    flag = ""
    if normalized_variance < VARIANCE_THRESHOLD:
        flag = "low variance"
    if dominant_ratio >= DOMINANT_VALUE_THRESHOLD:
        flag = (flag + " + dominant value").strip(" +") if flag else "dominant value"

    variance_report.append((col, round(normalized_variance, 4), round(dominant_ratio, 4), flag))

variance_df = pd.DataFrame(
    variance_report,
    columns=["feature", "normalized_variance", "dominant_value_ratio", "flag"]
)
variance_df = variance_df.sort_values("dominant_value_ratio", ascending=False)

flagged = variance_df[variance_df["flag"] != ""]
print(f"Flagged {len(flagged)} of {len(variance_df)} features as low-information:")
print(flagged.to_string(index=False))

low_info_cols = flagged["feature"].tolist()
further_reduced_df = reduced_numeric_df.drop(columns=low_info_cols)
print(f"\nFeatures before this step: {reduced_numeric_df.shape[1]}")
print(f"Features after this step:  {further_reduced_df.shape[1]}")




Flagged 8 of 57 features as low-information:
                       feature  normalized_variance  dominant_value_ratio                          flag
        handshake_ciphersuites               0.0000                1.0000 low variance + dominant value
                  ntp_interval             107.3552                0.9983                dominant value
    handshake_sig_hash_alg_len              22.8017                0.9978                dominant value
handshake_cipher_suites_length              21.6270                0.9970                dominant value
   handshake_extensions_length              22.4188                0.9970                dominant value
                icmp_data_size               2.8020                0.9946                dominant value
                   dns_len_ans              16.2855                0.9908                dominant value
                     icmp_type               0.7022                0.9901                dominant value

Features before th

In [7]:
# ============================================================
# Cell 6 — Check flagged features per attack type before dropping
#
# A feature can look "low variance / dominant value" overall while
# still being meaningful for one specific attack type (e.g.
# dns_len_ans is only relevant to DNS traffic — it'll look constant
# across the whole sample even if it's a strong signal specifically
# for DNS Spoofing rows).
#
# This uses `data["source_label"]` from Cell 1 to check each flagged
# feature's dominant-value ratio SEPARATELY per traffic type, instead
# of relying on the overall (misleading) ratio from Cell 5.
# ============================================================
per_label_report = []
for col in low_info_cols:
    for label in data["source_label"].unique():
        subset = data.loc[data["source_label"] == label, col].dropna()
        if subset.empty:
            continue
        dominant_ratio = subset.value_counts(normalize=True).iloc[0]
        per_label_report.append((col, label, round(dominant_ratio, 4)))

per_label_df = pd.DataFrame(
    per_label_report, columns=["feature", "source_label", "dominant_value_ratio"]
)

# For each feature, show the LOWEST dominant-value ratio across all
# attack types — i.e. the traffic type where this feature is LEAST
# constant, and therefore most likely to actually carry signal there.
most_informative_per_feature = (
    per_label_df.loc[per_label_df.groupby("feature")["dominant_value_ratio"].idxmin()]
    .sort_values("dominant_value_ratio")
)

print("Where each flagged feature is LEAST constant (most likely useful):")
print(most_informative_per_feature.to_string(index=False))

# Features worth keeping despite being flagged: ones that are clearly
# NOT constant for at least one attack type (dominant ratio well
# below the overall DOMINANT_VALUE_THRESHOLD).
KEEP_ANYWAY_THRESHOLD = 0.9
worth_keeping = most_informative_per_feature.loc[
    most_informative_per_feature["dominant_value_ratio"] < KEEP_ANYWAY_THRESHOLD, "feature"
].tolist()

print(f"\nFlagged features worth keeping anyway (protocol-specific signal): {worth_keeping}")

final_cols_to_drop = [c for c in low_info_cols if c not in worth_keeping]
final_reduced_df = reduced_numeric_df.drop(columns=final_cols_to_drop)
print(f"\nFeatures before this check: {reduced_numeric_df.shape[1]}")
print(f"Features after this check:  {final_reduced_df.shape[1]}")

Where each flagged feature is LEAST constant (most likely useful):
                       feature source_label  dominant_value_ratio
                     icmp_type          xss                0.9670
                   dns_len_ans       benign                0.9795
                icmp_data_size          xss                0.9848
handshake_cipher_suites_length       benign                0.9935
   handshake_extensions_length       benign                0.9935
                  ntp_interval          xss                0.9947
    handshake_sig_hash_alg_len          xss                0.9949
        handshake_ciphersuites       benign                1.0000

Flagged features worth keeping anyway (protocol-specific signal): []

Features before this check: 57
Features after this check:  49


In [8]:

# Cell 7 — PCA on the final feature set
#
# Steps:
#   1. Handle missing values (PCA can't handle NaNs) — using median
#      imputation here as a simple, defensible default. If you've
#      already got a more deliberate missing-value strategy from
#      Task 1.2, swap it in here instead.
#   2. Standardize (mean 0, std 1) — required because PCA is scale-
#      sensitive, and our features are on very different scales
#      (e.g. port-like values vs. entropy values vs. counts).
#   3. Fit PCA and look at how much variance each component explains,
#      so we can decide how many components to keep.
# ============================================================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
 
# 1. Impute missing values
imputer = SimpleImputer(strategy="median")
imputed = imputer.fit_transform(final_reduced_df)
 
# 2. Standardize
scaler = StandardScaler()
scaled = scaler.fit_transform(imputed)
 
# 3. Fit PCA with ALL components first, just to inspect variance curve
#    (we're not committing to a number of components yet)
pca_full = PCA()
pca_full.fit(scaled)
 
explained = pca_full.explained_variance_ratio_
cumulative = explained.cumsum()
 
variance_summary = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(explained))],
    "explained_variance_ratio": explained.round(4),
    "cumulative_variance": cumulative.round(4),
})
 
print(variance_summary.head(20).to_string(index=False))
 
# How many components needed to reach common variance thresholds
for target in [0.90, 0.95, 0.99]:
    n_components = (cumulative >= target).argmax() + 1
    print(f"Components needed for {int(target*100)}% variance: {n_components}")
 

component  explained_variance_ratio  cumulative_variance
      PC1                    0.2456               0.2456
      PC2                    0.1041               0.3497
      PC3                    0.0893               0.4390
      PC4                    0.0544               0.4934
      PC5                    0.0402               0.5336
      PC6                    0.0344               0.5680
      PC7                    0.0334               0.6013
      PC8                    0.0323               0.6336
      PC9                    0.0272               0.6608
     PC10                    0.0252               0.6861
     PC11                    0.0234               0.7095
     PC12                    0.0201               0.7295
     PC13                    0.0197               0.7493
     PC14                    0.0191               0.7684
     PC15                    0.0187               0.7871
     PC16                    0.0182               0.8052
     PC17                    0.

/Users/abhulimen/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning: divide by zero encountered in matmul
  C = X.T @ X
/Users/abhulimen/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning: overflow encountered in matmul
  C = X.T @ X
/Users/abhulimen/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_pca.py:606: RuntimeWarning: invalid value encountered in matmul
  C = X.T @ X


In [9]:
# ============================================================
# Cell 8 — Final selected feature list (for reuse elsewhere)
#
# This is the end result of Cells 1-6: 49 features, arrived at via
# correlation-based redundancy removal (window-size-aware) plus a
# protocol-aware low-variance check. PCA was tested in Cell 7 but
# not adopted (limited additional compression, not worth the loss
# of interpretability).
#
# Save this list so it can be reused directly in later notebooks
# (e.g. Task 2.1 Isolation Forest) without re-running the full
# selection pipeline each time.
# ============================================================
FINAL_SELECTED_FEATURES = sorted(final_reduced_df.columns.tolist())
 
print(f"Final selected feature count: {len(FINAL_SELECTED_FEATURES)}\n")
for i, feature in enumerate(FINAL_SELECTED_FEATURES, start=1):
    print(f"{i:2d}. {feature}")
 
import json
with open("final_selected_features.json", "w") as f:
    json.dump(FINAL_SELECTED_FEATURES, f, indent=2)
 
print("\nSaved to final_selected_features.json")
 


Final selected feature count: 49

 1. channel_10_var
 2. channel_1_mean
 3. channel_30_count
 4. channel_5_count
 5. dns_interval
 6. dns_len_qry
 7. dns_query_type
 8. eth_size
 9. http_content_len
10. http_response_code
11. inter_arrival_time
12. iqr_p
13. jitter
14. l3_ip_dst_count
15. l4_tcp
16. l4_udp
17. max_p
18. med_p
19. min_et
20. most_freq_spot
21. payload_entropy
22. port_class_dst
23. q1
24. src_ip_1_count
25. src_ip_1_mean
26. src_ip_1_var
27. src_ip_30_count
28. src_ip_30_var
29. src_ip_5_count
30. src_ip_5_var
31. stream_1_count
32. stream_1_mean
33. stream_1_var
34. stream_30_count
35. stream_30_mean
36. stream_5_count
37. stream_5_var
38. stream_jitter_1_mean
39. stream_jitter_1_sum
40. stream_jitter_30_sum
41. stream_jitter_5_sum
42. stream_jitter_5_var
43. stream_jitter_60_sum
44. stream_jitter_60_var
45. sum_p
46. tcp_window_size
47. time_since_previously_displayed_frame
48. ttl
49. var_e

Saved to final_selected_features.json


In [11]:
import json

with open("final_selected_features.json") as f:
    selected_features = json.load(f)

label_map = {
    "benign": "Benign", "benign_1": "Benign", "benign_2": "Benign", "benign_3": "Benign",
    "ddos": "DDoS", "dos": "DoS", "dos_1": "DoS",
    "dns_spoofing": "DNS_Spoofing", "brute_force": "BruteForce", "xss": "XSS",
}
data["Label"] = data["source_label"].map(label_map)

export_df = data[selected_features + ["Label"]].copy()
export_df[selected_features] = export_df[selected_features].replace([np.inf, -np.inf], np.nan).fillna(0)

print(export_df.shape)
print(export_df["Label"].value_counts())

export_df.to_csv("data/selected_features.csv", index=False)
print("saved to data/selected_features.csv")

(100000, 50)
Label
Benign          40000
DoS             20000
DDoS            10000
DNS_Spoofing    10000
BruteForce      10000
XSS             10000
Name: count, dtype: int64
saved to data/selected_features.csv
